# Model Monitoring & CloudWatch Dashboard
This notebook operationalizes the project monitoring requirements. We will implement **Data Quality**, **Model Quality**, and **Infrastructure** monitors, and consolidate them into a unified **CloudWatch Dashboard**.

In [1]:
!pip install "sagemaker==2.214.0"

In [2]:
import os
import boto3
import sagemaker
import pandas as pd
import time
from datetime import datetime, timedelta
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    EndpointInput,
    DatasetFormat
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name
prefix = "nyc-collisions-monitoring"

print(f"Monitoring initialized in {region} for bucket {bucket}")

## 1. Deploy Real-Time Endpoint
To use SageMaker Model Monitor, we need a live endpoint with **Data Capture** enabled. This captures all inference requests and saves them to S3 for analysis.

In [3]:
#  Define S3 Capture Path
data_capture_uri = f"s3://{bucket}/{prefix}/datacapture"

# Dynamically fetch the latest model from the project prefix
sm_client = boto3.client("sagemaker")
training_jobs = sm_client.list_training_jobs(
    NameContains="sagemaker-scikit-learn",
    StatusEquals="Completed",
    SortBy="CreationTime",
    SortOrder="Descending"
)

if training_jobs["TrainingJobSummaries"]:
    latest_job_name = training_jobs["TrainingJobSummaries"][0]["TrainingJobName"]
    model_artifact = sm_client.describe_training_job(TrainingJobName=latest_job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"Found latest model: {model_artifact}")
else:
    raise ValueError("No completed training jobs found to deploy.")

# Create Model Entity
image_uri = sagemaker.image_uris.retrieve(framework="sklearn", version="1.2-1", region=region)
model = Model(
    image_uri=image_uri, 
    model_data=model_artifact, 
    role=role,
    entry_point='sagemaker_train.py',
    source_dir='../src'
)

# Setup Data Capture
capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=data_capture_uri
)

# Deploy
endpoint_name = f"collisions-monitor-ep-{int(time.time())}"
model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=capture_config
)

predictor = Predictor(endpoint_name=endpoint_name, serializer=CSVSerializer())
print(f"Endpoint {endpoint_name} is LIVE with Data Capture.")

## 2. Data Quality Monitoring
We will baseline the validation data to identify **Data Drift** (e.g., if Borough distributions change).

In [4]:
data_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

# Use the validation split from Notebook 04
training_prefix = "aai-540-group6/nyc-collisions-ml"
baseline_data_uri = f"s3://{bucket}/{training_prefix}/validation/val.csv"
print(f"Using baseline data: {baseline_data_uri}")

data_monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/data_quality_results",
    wait=True
)

## 3. Model Quality Monitoring
We will track **Precision, Recall, and Accuracy** by comparing endpoint predictions against ground truth.

In [5]:
import botocore

model_sched_name = f"{endpoint_name}-model-qc"
schedule_already_exists = False
try:
    sm_client.describe_monitoring_schedule(MonitoringScheduleName=model_sched_name)
    schedule_already_exists = True
    print(f"Schedule {model_sched_name} already exists. Nothing to do.")
except botocore.exceptions.ClientError:
    pass

if not schedule_already_exists:
    print("--- Generating predictions for Model Quality baseline ---")
    df_val_local = pd.read_csv("data_splits/val.csv")
    sample_df = df_val_local.sample(250, random_state=42)
    features_only = sample_df.drop("target", axis=1)
    
    preds = []
    for _, row in features_only.iterrows():
        payload = ",".join([str(val) for val in row.values])
        response = predictor.predict(payload)
        clean_resp = response.decode("utf-8").strip().replace("[", "").replace("]", "")
        preds.append(int(float(clean_resp)))
    
    baseline_df = pd.DataFrame({"prediction": preds, "target": sample_df["target"].values})
    
    baseline_local_path = "data_splits/baseline_with_predictions.csv"
    baseline_df.to_csv(baseline_local_path, index=False)
    baseline_s3_uri = sess.upload_data(baseline_local_path, bucket, f"{prefix}/model_quality_baseline_with_preds")
    print(f"Uploaded combined baseline to: {baseline_s3_uri}")

    model_quality_monitor = ModelQualityMonitor(
        max_runtime_in_seconds=1800, # Less than 1 hour (3600s)
        role=role,
        instance_count=1,
        instance_type="ml.m5.large",
        sagemaker_session=sess
    )
    
    model_quality_monitor.suggest_baseline(
        baseline_dataset=baseline_s3_uri,
        dataset_format=sagemaker.model_monitor.dataset_format.DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{bucket}/{prefix}/model_quality_results",
        problem_type="BinaryClassification",
        ground_truth_attribute="target",
        inference_attribute="prediction",
        wait=True
    )

    print(f"\n--- Creating Model Quality Schedule: {model_sched_name} ---")
    model_constraints_s3_path = model_quality_monitor.latest_baselining_job.suggested_constraints().file_s3_uri
    
    model_quality_monitor.create_monitoring_schedule(
        monitor_schedule_name=model_sched_name,
        endpoint_input=EndpointInput(endpoint_name=endpoint_name, destination="/opt/ml/processing/input_data", inference_attribute="0"),
        problem_type="BinaryClassification",
        ground_truth_input=f"s3://{bucket}/{prefix}/ground_truth_data",
        constraints=model_constraints_s3_path,
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True
    )


## 4. CloudWatch Dashboard Implementation
We will now programmatically create a CloudWatch dashboard that tracks:
1. **Infrastructure:** Endpoint CPU & Memory.
2. **Data Quality:** Baseline violations.
3. **Model Quality:** Accuracy & Recall metrics.

In [6]:
import json
dashboard_body = {
    "widgets": [
        {
            "type": "metric", "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [["AWS/SageMaker", "CPUUtilization", "EndpointName", endpoint_name, "VariantName", "AllTraffic"]],
                "period": 300, "stat": "Average", "region": region,
                "title": "Infrastructure: CPU Utilization"
            }
        },
        {
            "type": "metric", "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [["AWS/SageMaker", "MemoryUtilization", "EndpointName", endpoint_name, "VariantName", "AllTraffic"]],
                "period": 300, "stat": "Average", "region": region,
                "title": "Infrastructure: Memory Utilization"
            }
        },
        {
            "type": "metric", "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["aws/sagemaker/Endpoints/model-metrics", "accuracy", "Endpoint", endpoint_name],
                    [".", "recall", ".", "."]
                ],
                "period": 3600, "stat": "Average", "region": region,
                "title": "Model Quality: Accuracy & Recall"
            }
        },
        {
            "type": "metric", "x": 12, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [["aws/sagemaker/Endpoints/data-metrics", "feature_baseline_drift_total_violations", "Endpoint", endpoint_name]],
                "period": 3600, "stat": "Sum", "region": region,
                "title": "Data Quality: Feature Drift Violations"
            }
        }
    ]
}

cw_client = boto3.client('cloudwatch')
cw_client.put_dashboard(
    DashboardName='NYC-Collision-ML-Performance',
    DashboardBody=json.dumps(dashboard_body)
)

print("CloudWatch Dashboard 'NYC-Collision-ML-Performance' created/updated successfully.")

In [7]:
# --- Simulate Traffic and Ground Truth for Monitoring ---
import random
import time
from threading import Thread
import json
import pandas as pd
import sagemaker
from datetime import datetime

# Thread to simulate endpoint traffic
def invoke_endpoint_forever():
    print("Starting traffic generator...")
    df_val_local = pd.read_csv("data_splits/val.csv")
    features_only = df_val_local.drop("target", axis=1)
    
    i = 0
    while True:
        try:
            # Pick a random row
            row = features_only.sample(1).iloc[0]
            payload = ",".join([str(val) for val in row.values])
            
            # Important: We must pass an InferenceId so Model Monitor can match it with Ground Truth later
            predictor.predict(data=payload, initial_args={"InferenceId": str(i)})
            i += 1
            time.sleep(1) # Send 1 request per second
        except Exception as e:
            pass

# Thread to simulate Ground Truth uploads
def generate_fake_ground_truth_forever():
    print("Starting Ground Truth generator...")
    j = 0
    while True:
        # Generate 300 fake ground truth records for the last 300 invocations
        fake_records = []
        for i in range(j, j + 300):
            # Randomly guess 0 or 1 for the "actual" outcome
            record = {
                "groundTruthData": {"data": str(random.choice([0, 1])), "encoding": "CSV"},
                "eventMetadata": {"eventId": str(i)},
                "eventVersion": "0",
            }
            fake_records.append(json.dumps(record))
            
        data_to_upload = "\n".join(fake_records)
        target_s3_uri = f"s3://{bucket}/{prefix}/ground_truth_data/{datetime.utcnow():%Y/%m/%d/%H/%M%S}.jsonl"
        sagemaker.s3.S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)
        
        j += 300
        time.sleep(60 * 60) # Upload once an hour

# Start the threads in the background
traffic_thread = Thread(target=invoke_endpoint_forever)
traffic_thread.start()

gt_thread = Thread(target=generate_fake_ground_truth_forever)
gt_thread.start()

print("Traffic and Ground Truth generators are running in the background.")


In [8]:
# Cleanup: Uncomment to delete endpoint and avoid costs
# predictor.delete_endpoint()
# predictor.delete_model()